In [4]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from cargar_datos import cargar_dataset_abandono_correcto

# Modelos de scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)

# === MODELOS A COMPARAR ===
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier  # Requiere: pip install lightgbm

# Preprocesamiento
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Todas las librerías cargadas")

✅ Todas las librerías cargadas


In [5]:
# Cargar dataset
df = cargar_dataset_abandono_correcto()

# Separar features y target
columnas_a_eliminar = ['id_reserva', 'id_cliente', 'fecha_inicio_reserva', 'es_abandono']
X = df.drop(columns=columnas_a_eliminar, errors='ignore')
y = df['es_abandono']

# One-Hot Encoding
columnas_texto = X.select_dtypes(include=['object']).columns.tolist()
if columnas_texto:
    X = pd.get_dummies(X, columns=columnas_texto, drop_first=True)

# Convertir booleanos
X = X.astype({col: 'int8' for col in X.select_dtypes(include=['bool']).columns})
X = X.fillna(0)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Escalado (necesario para algunos modelos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Dataset listo: {X_train.shape[0]} train, {X_test.shape[0]} test")
print(f"📊 Features: {X.shape[1]}")

📊 CARGANDO DATASET DE ABANDONO
✅ Conexión exitosa a SQL Server
✅ Consulta ejecutada: 10,786 registros, 27 columnas

📏 Dimensiones: 10,786 filas × 27 columnas

🎯 Distribución de la variable target:
es_abandono
1    6551
0    4235
Name: count, dtype: int64

📈 Tasa de abandono: 60.74%
✅ Dataset listo: 8628 train, 2158 test
📊 Features: 27


In [6]:
# Ratio de desbalance
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# Diccionario con TODOS los modelos
modelos = {
    '1. Regresión Logística': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Maneja desbalance
        random_state=42
    ),
    
    '2. Árbol de Decisión': DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42
    ),
    
    '3. Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    
    '4. XGBoost': XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        eval_metric='auc',
        verbosity=0
    ),
    
    '5. LightGBM': LGBMClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=ratio,
        random_state=42,
        verbose=-1
    ),
    
    '6. KNN (k=10)': KNeighborsClassifier(
        n_neighbors=10,
        weights='distance',  # Pesa más a los más cercanos
        n_jobs=-1
    ),
    
    '7. Naive Bayes': GaussianNB(),
    
    '8. SVM (RBF)': SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced',
        random_state=42
    ),
}

print(f"✅ {len(modelos)} modelos configurados")

✅ 8 modelos configurados


In [7]:
resultados = []

print("=" * 80)
print("🤖 ENTRENANDO Y EVALUANDO TODOS LOS MODELOS")
print("=" * 80)

for nombre, modelo in modelos.items():
    print(f"\n🔄 {nombre}...")
    inicio = time.time()
    
    try:
        # Algunos modelos necesitan datos escalados
        if nombre in ['1. Regresión Logística', '6. KNN (k=10)', '8. SVM (RBF)']:
            X_tr, X_te = X_train_scaled, X_test_scaled
        else:
            X_tr, X_te = X_train, X_test
        
        # Entrenar
        modelo.fit(X_tr, y_train)
        
        # Predecir
        y_pred = modelo.predict(X_te)
        y_pred_proba = modelo.predict_proba(X_te)[:, 1]
        
        # Métricas
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)
        
        # Cross-validation (5-fold)
        cv_scores = cross_val_score(modelo, X_tr, y_train, cv=5, scoring='roc_auc')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        # Tiempo
        tiempo = time.time() - inicio
        
        # Guardar resultados
        resultados.append({
            'Modelo': nombre,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': auc,
            'CV ROC-AUC': cv_mean,
            'CV Std': cv_std,
            'Tiempo (s)': tiempo
        })
        
        print(f"   ✅ Accuracy: {accuracy:.3f} | AUC: {auc:.3f} | CV-AUC: {cv_mean:.3f}±{cv_std:.3f}")
        print(f"   ⏱️ Tiempo: {tiempo:.2f}s")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        resultados.append({
            'Modelo': nombre,
            'Accuracy': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0,
            'ROC-AUC': 0, 'CV ROC-AUC': 0, 'CV Std': 0, 'Tiempo (s)': 0
        })

# DataFrame con resultados
df_resultados = pd.DataFrame(resultados).sort_values('ROC-AUC', ascending=False)

🤖 ENTRENANDO Y EVALUANDO TODOS LOS MODELOS

🔄 1. Regresión Logística...
   ✅ Accuracy: 0.656 | AUC: 0.754 | CV-AUC: 0.755±0.012
   ⏱️ Tiempo: 0.11s

🔄 2. Árbol de Decisión...
   ✅ Accuracy: 0.678 | AUC: 0.774 | CV-AUC: 0.758±0.009
   ⏱️ Tiempo: 0.25s

🔄 3. Random Forest...
   ✅ Accuracy: 0.707 | AUC: 0.814 | CV-AUC: 0.799±0.008
   ⏱️ Tiempo: 2.62s

🔄 4. XGBoost...
   ✅ Accuracy: 0.724 | AUC: 0.820 | CV-AUC: 0.817±0.004
   ⏱️ Tiempo: 4.12s

🔄 5. LightGBM...
   ❌ Error: exception: access violation reading 0x0000000000000000

🔄 6. KNN (k=10)...
   ✅ Accuracy: 0.741 | AUC: 0.830 | CV-AUC: 0.823±0.009
   ⏱️ Tiempo: 0.17s

🔄 7. Naive Bayes...
   ✅ Accuracy: 0.652 | AUC: 0.729 | CV-AUC: 0.724±0.015
   ⏱️ Tiempo: 0.05s

🔄 8. SVM (RBF)...
   ✅ Accuracy: 0.676 | AUC: 0.777 | CV-AUC: 0.764±0.012
   ⏱️ Tiempo: 38.71s


In [8]:
print("=" * 100)
print("📊 TABLA COMPARATIVA DE MODELOS")
print("=" * 100)

# Formatear para visualización
df_display = df_resultados.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'CV ROC-AUC']:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")
df_display['CV Std'] = df_display['CV Std'].apply(lambda x: f"±{x:.4f}")
df_display['Tiempo (s)'] = df_display['Tiempo (s)'].apply(lambda x: f"{x:.2f}")

print(df_display.to_string(index=False))

# Destacar el mejor modelo
mejor_modelo = df_resultados.iloc[0]['Modelo']
mejor_auc = df_resultados.iloc[0]['ROC-AUC']

print(f"\n🏆 MEJOR MODELO: {mejor_modelo}")
print(f"   ROC-AUC: {mejor_auc:.4f}")

📊 TABLA COMPARATIVA DE MODELOS
                Modelo Accuracy Precision Recall F1-Score ROC-AUC CV ROC-AUC  CV Std Tiempo (s)
         6. KNN (k=10)   0.7414    0.7850 0.7910   0.7880  0.8304     0.8233 ±0.0087       0.17
            4. XGBoost   0.7243    0.8352 0.6804   0.7499  0.8201     0.8173 ±0.0043       4.12
      3. Random Forest   0.7067    0.8725 0.6056   0.7150  0.8144     0.7988 ±0.0084       2.62
          8. SVM (RBF)   0.6756    0.8524 0.5637   0.6786  0.7771     0.7641 ±0.0116      38.71
  2. Árbol de Decisión   0.6779    0.8468 0.5736   0.6839  0.7745     0.7584 ±0.0089       0.25
1. Regresión Logística   0.6557    0.7977 0.5805   0.6720  0.7535     0.7546 ±0.0122       0.11
        7. Naive Bayes   0.6520    0.8011 0.5683   0.6649  0.7285     0.7239 ±0.0149       0.05
           5. LightGBM   0.0000    0.0000 0.0000   0.0000  0.0000     0.0000 ±0.0000       0.00

🏆 MEJOR MODELO: 6. KNN (k=10)
   ROC-AUC: 0.8304
